# Capítulo 4: Aprendizaje por Refuerzo Tradicional: Q-Learning en el Laberinto

En este cuaderno implementamos el algoritmo clásico de **Q-Learning** (Watkins, 1989) para entrenar al agente **OptiBot** en una cuadrícula de $3 \times 3$.

### Entorno y Reglas:
- Casillas: $S_1$ (Inicio) a $S_9$ (Meta).
- Muro central infranqueable: $S_5$.
- Recompensa de paso normal: $R = -1.0$.
- Salirse de la cuadrícula: Penalización de $R = -10.0$ y fin de episodio (*Game Over*).
- Meta ($S_9$): $R = +10.0$ (estado terminal).
- Regla de actualización:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \Big[ R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t) \Big]$$


In [ ]:
import numpy as np

class LaberintoOptiBot:
    """
    Cuadrícula 3x3 del almacén:
      S1(0) S2(1) S3(2)
      S4(3) S5(4:Muro) S6(5)
      S7(6) S8(7) S9(8:Meta)
    Acciones: 0: Norte (^), 1: Sur (v), 2: Este (>), 3: Oeste (<)
    """
    def __init__(self):
        self.n_estados = 9
        self.n_acciones = 4
        self.muro = 4       # S5 está bloqueado
        self.meta = 8       # S9 es la meta
        self.estado = 0     # Inicia en S1

    def reset(self):
        self.estado = 0
        return self.estado

    def step(self, accion):
        r, c = divmod(self.estado, 3)
        nr, nc = r, c
        salio = False

        if accion == 0:    # Norte
            if r > 0: nr -= 1
            else: salio = True
        elif accion == 1:  # Sur
            if r < 2: nr += 1
            else: salio = True
        elif accion == 2:  # Este
            if c < 2: nc += 1
            else: salio = True
        elif accion == 3:  # Oeste
            if c > 0: nc -= 1
            else: salio = True

        # Salirse del laberinto provoca Game Over con penalización de -10
        if salio:
            return self.estado, -10.0, True

        s_sig = nr * 3 + nc
        # Si choca contra el muro central, rebota y no se mueve
        if s_sig == self.muro:
            s_sig = self.estado

        # Evaluación de llegada a la Meta
        if s_sig == self.meta:
            recompensa = 10.0
            done = True
        else:
            recompensa = -1.0
            done = False

        self.estado = s_sig
        return s_sig, recompensa, done


class AgenteQLearning:
    def __init__(self, n_estados=9, n_acciones=4, alfa=0.20, gamma=0.90, epsilon=1.0):
        self.Q = np.zeros((n_estados, n_acciones))
        self.alfa = alfa
        self.gamma = gamma
        self.epsilon = epsilon
        self.eps_min = 0.05
        self.decaimiento = 0.995

    def elegir_accion(self, s):
        # Exploración vs Explotación
        if np.random.rand() < self.epsilon:
            return np.random.randint(4)
        return int(np.argmax(self.Q[s]))

    def actualizar(self, s, a, r, s_sig, done):
        # Regla de Watkins
        q_futuro = 0.0 if done else np.max(self.Q[s_sig])
        objetivo_td = r + self.gamma * q_futuro
        error_td = objetivo_td - self.Q[s, a]
        self.Q[s, a] += self.alfa * error_td

    def decaer_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.decaimiento)


## Entrenamiento del Agente con Q-Learning


In [ ]:
np.random.seed(42)
env = LaberintoOptiBot()
agente = AgenteQLearning()

for ep in range(1, 2001):
    s = env.reset()
    done = False
    pasos = 0
    while not done and pasos < 50:
        a = agente.elegir_accion(s)
        s_sig, r, done = env.step(a)
        agente.actualizar(s, a, r, s_sig, done)
        s = s_sig
        pasos += 1
    agente.decaer_epsilon()

simbolos = {0: "^", 1: "v", 2: ">", 3: "<"}
print("=== POLÍTICA ÓPTIMA APRENDIDA POR OPTIBOT ===")
for r in range(3):
    fila_str = ""
    for c in range(3):
        s = r * 3 + c
        if s == env.meta:
            fila_str += "[ META ] "
        elif s == env.muro:
            fila_str += "[ MURO ] "
        else:
            mejor_a = np.argmax(agente.Q[s])
            fila_str += f"[  {simbolos[mejor_a]}   ] "
    print(fila_str)


## Tabla Q Aprendida vs Tabla Teórica $Q^*(s, a)$


In [ ]:
nombres_estados = [
    "S1 (Inicio)", "S2 (Pasillo)", "S3 (Pasillo)",
    "S4 (Pasillo)", "S5 (Muro)   ", "S6 (Pasillo)",
    "S7 (Pasillo)", "S8 (Pasillo)", "S9 (Meta)   "
]
acciones_nombres = ["Norte (^)", "Sur (v)", "Este (>)", "Oeste (<)"]

print(f"{'Estado':14s} | " + " | ".join(f"{a:9s}" for a in acciones_nombres))
print("-" * 65)
for s in range(9):
    if s == 4:
        print(f"{nombres_estados[s]:14s} |    ---    |    ---    |    ---    |    ---   ")
    else:
        fila = " | ".join(f"{agente.Q[s, a]:+9.2f}" for a in range(4))
        print(f"{nombres_estados[s]:14s} | {fila}")
